<p align="center">
  <img src="./assets/banners/99-learner-guide.svg" alt="MAI Models · Multimodal Product Launch — Learner guide: How do you build with MAI models, step by step? Microsoft AI Samples" width="900" />
</p>

# 📘 Learner guide: building with MAI models

> **One manual for the whole workshop: what each MAI model does, how your code talks to it, and how to fix it when it doesn't answer.**

⏱️ About 30 minutes to read · 🧪 Every code cell runs **without Azure**: it builds requests and checks them, but never sends them · 🧱 You'll leave with: a mental model, request templates for each model, and a debugging playbook

This guide sits beside notebooks 00 to 04. They tell the story of the TrailMaster X4 launch; this guide is the reference you keep open while you build your own.

**How this guide teaches**

| Icon | Step | What you do |
| --- | --- | --- |
| 🔮 | **Predict** | Guess what a request will do before you see it |
| 🧱 | **Construct** | Build the request one piece at a time |
| 🐞 | **Debug** | Break it on purpose, read the error, fix it |
| 👀 | **Observe** | Look at what the model really did in our run |

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 0. Your route through the guide

| # | Section | The question it answers | Model |
| --- | --- | --- | --- |
| 1 | [Meet the MAI models](#1-meet-the-mai-models) | Which model does which job? | All |
| 2 | [The mental model](#2-the-mental-model-one-resource-three-doors) | How does my code reach a model? | All |
| 3 | [Build with images](#3-build-with-images-mai-image-26) | How do I make and edit pictures? | MAI-Image-2.6 |
| 4 | [Build with voice](#4-build-with-voice-mai-voice-21) | How do I turn text into speech? | MAI-Voice-2.1 |
| 5 | [Build with transcription](#5-build-with-transcription-mai-transcribe-2) | How do I turn speech into checked text? | MAI-Transcribe-2 |
| 6 | [Beyond your resource](#6-beyond-your-resource-code-thinking-and-cyber) | Where do the other MAI models live? | Code, Thinking, Cyber |
| 7 | [The debugging playbook](#7-the-debugging-playbook) | It failed. What now? | All |
| 8 | [Pick the right model](#8-pick-the-right-model) | Standard or Flash? | All |
| 9 | [Check yourself](#9-check-yourself) | Did it stick? | All |

Run the next cell first. It finds this folder and the shared product data, and every later cell uses them.

In [1]:
import json
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "contoso-retail").is_dir())
HERE = ROOT / "workshops" / "mai-multimodal-product-launch"
DATA = ROOT / "data" / "contoso-retail"

products = json.loads((DATA / "products.json").read_text(encoding="utf-8"))
tent = next(p for p in products if p["slug"] == "trailmaster-x4-tent")
image_log = json.loads((HERE / "examples" / "01" / "run_log.json").read_text())

print(f"📦 {len(products)} products in {DATA.relative_to(ROOT)}/ · our hero: {tent['name']} (${tent['price']:g}, {tent['brand']})")
print(f"🖼️ {len(image_log)} saved pictures from our MAI-Image run in examples/01/")

📦 20 products in data/contoso-retail/ · our hero: TrailMaster X4 Tent ($250, OutdoorLiving)
🖼️ 47 saved pictures from our MAI-Image run in examples/01/


<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 1. Meet the MAI models

Microsoft AI (MAI) builds its own foundation models. Each one has a clear job, and most come in two sizes: a **standard** model for the best result, and a **Flash** model that's faster and cheaper.

| Model | Its job | Where you use it | Sizes | Learn more |
| --- | --- | --- | --- | --- |
| **MAI-Image-2.6** | Create and precisely edit design-ready images from text and reference pictures | Microsoft Foundry · MAI Playground | Standard · Flash | [Page](https://microsoft.ai/models/mai-image-2-6/) · [Card (PDF)](../../.github/.mc/MAI-Image-2.6-Model-Card.pdf) |
| **MAI-Voice-2.1** | Expressive text-to-speech in 23 languages, long-form or live | Azure Speech SDK and REST, in Foundry | Standard · Flash | [Page](https://microsoft.ai/models/mai-voice-2-1/) · [Card](../../.github/.mc/MAI-Voice-2.1-Model-Card-Memo.pdf) · [Flash card](../../.github/.mc/MAI-Voice-2.1-Flash-Model-Card-Memo.pdf) |
| **MAI-Transcribe-2** | Speech-to-text in 60 languages, with speaker labels, word timestamps and keyword biasing | Azure Speech, in Foundry | Batch · Streaming | [Page](https://microsoft.ai/models/mai-transcribe-2/) · [Card](../../.github/.mc/MAI-Transcribe-2-Model-Card.pdf) · [Streaming card](../../.github/.mc/MAI-Transcribe-2-Streaming-Model-Card-Memo.pdf) |
| **MAI-Code-1.1-Flash** | Agentic coding in real repositories, including screenshot-to-prototype | GitHub Copilot in VS Code | Flash | [Page](https://microsoft.ai/models/mai-code-1-flash/) · [Card for 1-Flash](../../.github/.mc/MAI-Code-1-Flash-Model-Card.pdf) |
| **MAI-Thinking-1** | Reasoning for math, coding and enterprise work | Microsoft Foundry (public preview) | One size | [Page](https://microsoft.ai/models/mai-thinking-1/) |
| **MAI-Cyber-1.1-Flash** | Find, prioritize and fix software vulnerabilities | MDASH only, for verified defenders | Flash | [Page](https://microsoft.ai/models/mai-cyber-1-1-flash/) · [Card](../../.github/.mc/MAI-Cyber-1.1-Flash-ORA-Model-Card.pdf) |

This workshop builds with the first four. Section 6 covers where Thinking and Cyber fit.

> 📄 **Model cards are the canonical source.** Each card states what the model is, what it's for, what's out of scope, and how it was evaluated for safety. The facts below come from the cards linked above; where a card and the product disagree, the card's date is shown so you know which is newer.

### How the family fits together

The language models share one lineage. The model cards describe each step:

```text
MAI-Thinking-1 ── mid-training checkpoint ──▶ MAI-Code-1-Flash ──▶ MAI-Code-1.1-Flash
  reasoning, ~1T total / 35B active (MoE)        coding, 137B / 5B active      the model in Copilot today
                                                       │
                                                       └── fine-tune ──▶ MAI-Cyber-1-Flash ──▶ MAI-Cyber-1.1-Flash
                                                                                              fewer over-refusals, MDASH only
MAI-Image-2.6 ──▶ MAI-Image-2.6-Flash              (diffusion, 20B: an optimized version of 2.6)
MAI-Voice-2.1 ──▶ MAI-Voice-2.1-Flash              (text-to-speech, tuned for low latency)
MAI-Transcribe-2 ──▶ MAI-Transcribe-2-Streaming    (speech-to-text, live audio)
```

### Spec sheets

Open one to see what its model card says. Each ends with a **🐞 watch out** line: the spec that most often trips developers up.

<details>
<summary><b>🎨 MAI-Image-2.6 and MAI-Image-2.6-Flash</b> · <a href="../../.github/.mc/MAI-Image-2.6-Model-Card.pdf">model card</a> (4 Sep 2026)</summary>

| | |
| --- | --- |
| **What it is** | A 20B-parameter diffusion model for text-to-image and image-to-image editing. Flash is an optimized version of 2.6 |
| **Inputs → outputs** | Text, plus images for editing → one image |
| **Context** | 32K tokens of prompt |
| **Size limit** | At most **2,359,296 pixels** in total (1536 × 1536). Either side can go above 1536 if the total stays within the limit |
| **Strengths** | Precise, surgical edits that keep the layout: remove, replace, recolor, inpaint, update text, clean up motion blur. Photorealism, portraits, product and brand imagery, and text on pictures |
| **Released** | 2.6 on 14 Aug 2026, 2.6-Flash on 4 Sep 2026 |
| **Channels** | Microsoft Foundry (API) and MAI Playground. The card lists Foundry as private preview, but the Foundry catalog showed both models as generally available when this guide was written |
| **Out of scope** | Deceiving or impersonating real people, illegal content, harmful or policy-violating content. Prompt and output filters are applied |

🐞 **Watch out:** the limit is total pixels, not the longest side. 2304 × 1024 works; 1600 × 1600 doesn't (section 3).
</details>

<details>
<summary><b>🎙️ MAI-Voice-2.1 and MAI-Voice-2.1-Flash</b> · <a href="../../.github/.mc/MAI-Voice-2.1-Model-Card-Memo.pdf">card</a> · <a href="../../.github/.mc/MAI-Voice-2.1-Flash-Model-Card-Memo.pdf">Flash card</a> (1 Oct 2026)</summary>

| | MAI-Voice-2.1 | MAI-Voice-2.1-Flash |
| --- | --- | --- |
| **Built for** | Fidelity: audiobooks, narration, voice-over, marketing | Speed: call centers, IVR, voice assistants, live support |
| **Languages** | 23 languages, 26 locales. **One voice identity across every language** | 23 languages |
| **Control** | Emotion, tone and delivery per sentence, with SSML styles such as `excited` | Same, plus **streaming output with barge-in**, so a caller can interrupt |
| **Long-form and dialogue** | Long-form consistency through chunking and context carry-over · multiple speakers | Multiple speakers |
| **Voices** | A curated library of licensed voices (for example `Harper`, `Grant`), or **voice cloning** from a 5–60 second clip | Same |
| **Price** | $22 per 1M characters | $15 per 1M characters |
| **How you call it** | Speech SDK (.NET, Python, Java, JavaScript, C++) or REST. SSML voice name `en-US-Harper:MAI-Voice-2.1` | `en-US-Harper:MAI-Voice-2.1-Flash` |

🔐 **Voice cloning is gated.** You apply for limited access, upload a recorded consent statement from the voice talent, and only then synthesize. The workshop uses library voices only.

🐞 **Watch out:** change languages by changing the **locale prefix** of the voice name (`es-MX-Harper:MAI-Voice-2.1`), not the model. The same voice speaks every language.
</details>

<details>
<summary><b>👂 MAI-Transcribe-2 and MAI-Transcribe-2-Streaming</b> · <a href="../../.github/.mc/MAI-Transcribe-2-Model-Card.pdf">card</a> (3 Sep 2026) · <a href="../../.github/.mc/MAI-Transcribe-2-Streaming-Model-Card-Memo.pdf">Streaming card</a> (1 Oct 2026)</summary>

| | MAI-Transcribe-2 (batch) | MAI-Transcribe-2-Streaming |
| --- | --- | --- |
| **Built for** | Recorded audio: calls, meetings, clinical notes, subtitles, archives | Live audio: captions, voice agents, dictation, agent assist |
| **Languages** | 60, with automatic language detection | 60 |
| **Features** | Speaker diarization, word-level timestamps, keyword biasing, **verbatim or clean** style | Partial results that update as audio arrives, then a stable final result per utterance |
| **Speed** | About 10 seconds for 1 hour of audio | Text while the person is still talking |
| **Accuracy** | Lowest average WER on FLEURS across 60 languages; #2 on the Artificial Analysis English leaderboard | 2.5% WER on the Artificial Analysis leaderboard, the lowest listed |
| **Input** | WAV, MP3 or FLAC · up to **300 MB or 2 hours** per file | A continuous audio stream |
| **How you call it** | Azure Speech fast transcription REST (`transcriptions:transcribe`) | OpenAI Realtime-compatible WebSocket API, or the Speech SDK |
| **Price** | $0.10 per hour (limited-time offer) | $0.54 per hour (through 31 Dec 2026) |
| **Served from** | East US, West US, West US 2, North Europe, Central India, Southeast Asia | Central US, Sweden Central, Southeast Asia (East US 2 coming soon) |

🐞 **Watch out:** Streaming is **not** in East US, this workshop's default region. For live captions, deploy a second resource in a Streaming region.
</details>

<details>
<summary><b>💻 MAI-Code-1.1-Flash</b> · <a href="../../.github/.mc/MAI-Code-1-Flash-Model-Card.pdf">card for MAI-Code-1-Flash</a></summary>

The card in this repo describes **MAI-Code-1-Flash** (released 2 Jun 2026), the model 1.1-Flash builds on.

| | |
| --- | --- |
| **What it is** | A sparse mixture-of-experts coding model, **137B total / 5B active** parameters, built from MAI-Thinking-1's mid-training checkpoint |
| **Context** | 256K tokens · text in, text out · English |
| **Trained for** | Agentic coding in real repositories, repo Q&A, refactoring and tool use, **trained directly with the GitHub Copilot harness**. It keeps simple answers short and spends more reasoning on hard tasks |
| **Benchmarks (1-Flash)** | SWE-Bench Verified 71.6% · SWE-Bench Pro 51.2% · Terminal Bench 2 54.8%, measured end to end in the VS Code harness |
| **Channels** | GitHub Copilot in VS Code, through the model picker or the Auto picker. The 1.1 model page adds the Copilot CLI and on-device use |
| **Known limits** | Generated code can be wrong or incomplete. Review, test and validate before you rely on it |

🐞 **Watch out:** there's no public API and nothing to deploy. If you don't see it in the model picker, your Copilot plan or rollout stage may not include it yet.
</details>

<details>
<summary><b>🧠 MAI-Thinking-1</b> · no model card in this repo yet</summary>

From the [model page](https://microsoft.ai/models/mai-thinking-1/): Microsoft AI's first reasoning model, a mixture-of-experts model with about **35B active / ~1T total** parameters, built for math, coding and enterprise work, in **public preview** in Microsoft Foundry. It's also the starting point for MAI-Code-1-Flash.

🐞 **Watch out:** reasoning costs tokens and time. Use it for the hard step in a workflow, not every reply.
</details>

<details>
<summary><b>🛡️ MAI-Cyber-1.1-Flash</b> · <a href="../../.github/.mc/MAI-Cyber-1.1-Flash-ORA-Model-Card.pdf">model card</a></summary>

| | |
| --- | --- |
| **What it is** | A cybersecurity fine-tune of MAI-Cyber-1-Flash (itself built from MAI-Code-1-Flash): **137B total / 5B active**, 256K context |
| **Built for** | Vulnerability detection, prioritization and remediation across large codebases, inside **codename MDASH** |
| **What changed in 1.1** | Trained to reduce cyber over-refusals, for a wider range of agentic security workflows |
| **Access** | Foundry private preview, **only for use within MDASH**, for select verified customers after review. Users can't edit prompts, and use outside MDASH triggers an alert and revoked access |
| **Out of scope** | Anything other than defensive security operations within MDASH |
| **Safety** | Red-teamed internally and by an independent firm. The findings are why the model isn't offered for direct access |

🐞 **Watch out:** you can't call it from your own code. If your scenario needs a security model, start with MDASH.
</details>

### Standard or Flash, side by side

| | Standard | Flash or Streaming |
| --- | --- | --- |
| **Image** | MAI-Image-2.6: highest quality and the most precise edits | MAI-Image-2.6-Flash: production-ready quality at less than half the price |
| **Voice** | MAI-Voice-2.1: ~550 ms model inference, $22 per 1M characters. For content creation | MAI-Voice-2.1-Flash: ~45 ms model inference, $15 per 1M characters, streaming with barge-in. For call centers and assistants |
| **Transcribe** | MAI-Transcribe-2: 1 hour of audio in about 10 seconds, $0.10 per hour | MAI-Transcribe-2-Streaming: ~120 ms to first partial result, $0.54 per hour |

Latency figures come from the model pages; prices from the cards and pages. Check the [MAI pricing page](https://aka.ms/MAIModelPricing) before you quote them.

### Responsible use at a glance

| Model | What the card asks of you |
| --- | --- |
| Image | Don't deceive or impersonate real people. Label AI-generated pictures |
| Voice | Voice cloning needs approval and the speaker's recorded consent. Tell listeners the voice is AI |
| Transcribe | Measure accuracy on your own audio before you rely on it in clinical, legal or financial settings |
| Code | Review, test and validate generated code before production |
| Cyber | Defensive use only, inside MDASH |

> 🔮 **Predict:** the Voice Flash model is about 12 times faster at *model inference*. Will your listener hear the first word 12 times sooner? Keep your guess; section 4 measures it.

> 📰 **Versions move fast.** While this guide was being written, the MAI blog announced MAI-Image-2.7. The patterns here stay the same; only the model name in your deployment changes.

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 2. The mental model: one resource, three doors

<img src="./assets/guide/three-doors.svg" alt="One Microsoft Foundry resource with three doors: images, voice and transcribe. Code, Thinking and Cyber are used outside your resource." width="900" />

Everything in notebooks 01 to 03 goes through **one Microsoft Foundry resource**. Learn these four facts and most errors explain themselves:

1. **One resource, two addresses.** The resource's name is its subdomain. Images use `https://<name>.services.ai.azure.com`; speech uses `https://<name>.cognitiveservices.azure.com`. The notebooks work out both from `FOUNDRY_PROJECT_ENDPOINT`.
2. **Image models are deployed; speech models are built in.** You create deployments for MAI-Image-2.6 and Flash. MAI-Voice and MAI-Transcribe come with the resource, so you just name them in the request.
3. **No keys.** Your code asks Microsoft Entra ID for a token (`DefaultAzureCredential` uses your `az login`). The resource checks that you have the **Foundry User** role. `infra/main.bicep` turns keys off entirely.
4. **Region matters.** Not every region offers every MAI model. `eastus` offers image, voice and transcription; `westus` has no MAI voices; `westus3` has no MAI image models.

### 🧱 Construct: the sign-in every door shares

```python
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()   # Uses your az login. No secret in the code.
token = credential.get_token("https://cognitiveservices.azure.com/.default").token
headers = {"Authorization": f"Bearer {token}"}
```

The same token scope works for all three doors. Tokens last about an hour; `DefaultAzureCredential` refreshes them for you if you call `get_token()` per request, as the notebooks do.

### 🧱 Construct: the infrastructure, in one command

```bash
az login --use-device-code
bash infra/setup.sh            # Foundry resource + project, image deployments, Foundry User role, writes .env
bash infra/teardown.sh         # when you're done
```

> 🔮 **Predict:** you run `setup.sh`, then call the image API straight away and get **403**. Is the deployment broken? *(Answer in section 7.)*

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 3. Build with images: MAI-Image-2.6

<img src="./assets/asks/01-05.svg" alt="Your customer asks: Can it put real words on a picture? We always fix the text by hand afterwards." width="900" />

The image door has two endpoints:

| Endpoint | Use it to | Body |
| --- | --- | --- |
| `POST /mai/v1/images/generations` | Draw a new picture from words | JSON |
| `POST /mai/v1/images/edits` | Change a picture, or combine 1 to 5 reference pictures | Multipart form + images (PNG or JPEG) |

Both reply with the picture as base64 in `data[0].b64_json`.

### 🧱 Construct a generation request, one piece at a time

1. **Who:** `model` is your **deployment name**, not the catalog name. They're the same only if you named them that way (`setup.sh` does).
2. **What:** `prompt` is a description. Longer, specific briefs follow your brand better (notebook 01, section 1.6).
3. **Shape:** `width` and `height`. Each side must be at least **768**, and the total at most **2,359,296 pixels** (that's 1536 × 1536).
4. **Options:** `web_grounding` lets the model look things up first; `auto_aspect_ratio` lets it pick the shape.

The next cell builds requests and checks them the way the service will, without sending anything.

In [2]:
MAX_PIXELS = 2_359_296
MIN_SIDE = 768


def image_request(prompt: str, model: str = "MAI-Image-2.6", size: tuple[int, int] | None = (1024, 1024), **options) -> dict:
    """Build the JSON body for /mai/v1/images/generations, and refuse anything the service would reject."""
    body = {"model": model, "prompt": prompt, **options}
    if size:
        width, height = size
        if min(width, height) < MIN_SIDE:
            raise ValueError(f"400 waiting to happen: {width}×{height} has a side under {MIN_SIDE}.")
        if width * height > MAX_PIXELS:
            raise ValueError(f"400 waiting to happen: {width}×{height} = {width * height:,} pixels, over {MAX_PIXELS:,}.")
        body["width"], body["height"] = width, height
    return body


poster = image_request(f"A launch poster for the {tent['name']}. Headline: \"HOME, ANYWHERE.\" Price: \"${tent['price']:g}\".", size=(1024, 1536))
print("✅ Poster request:\n" + json.dumps(poster, indent=2))

✅ Poster request:
{
  "model": "MAI-Image-2.6",
  "prompt": "A launch poster for the TrailMaster X4 Tent. Headline: \"HOME, ANYWHERE.\" Price: \"$250\".",
  "width": 1024,
  "height": 1536
}


### 🐞 Debug: three sizes, one rule

🔮 **Predict:** which of these sizes will the service accept? A wide web banner at **2304 × 1024**, a square at **1600 × 1600**, and a small thumbnail at **512 × 512**.

In [3]:
for label, size in {"web banner": (2304, 1024), "big square": (1600, 1600), "thumbnail": (512, 512)}.items():
    try:
        body = image_request("A banner for the TrailMaster X4 Tent.", size=size)
        print(f"✅ {label:10} {size[0]}×{size[1]} = {size[0] * size[1]:>9,} pixels · accepted")
    except ValueError as problem:
        print(f"🐞 {label:10} {problem}")

print("\n🔧 Fix: 1536×1536 is the biggest square. For a thumbnail, generate at 768 or more, then resize it yourself.")

✅ web banner 2304×1024 = 2,359,296 pixels · accepted
🐞 big square 400 waiting to happen: 1600×1600 = 2,560,000 pixels, over 2,359,296.
🐞 thumbnail  400 waiting to happen: 512×512 has a side under 768.

🔧 Fix: 1536×1536 is the biggest square. For a thumbnail, generate at 768 or more, then resize it yourself.


### 🧱 Construct: the real call

This is the heart of `_call()` in notebook 01. Everything else there is replay, timing and rate limiting.

```python
import requests
from urllib.parse import urlparse

host = urlparse(FOUNDRY_PROJECT_ENDPOINT).hostname          # <name>.services.ai.azure.com
reply = requests.post(f"https://{host}/mai/v1/images/generations",
                      headers=headers, json=poster, timeout=300)
reply.raise_for_status()
png = base64.b64decode(reply.json()["data"][0]["b64_json"])
```

**Editing** sends a form instead, with the pictures attached. Form values are text, so `True` travels as `"true"`:

```python
files = [("image", ("campsite.png", open("campsite.png", "rb"), "image/png"))]
form = {"model": "MAI-Image-2.6", "prompt": "Change the price on the sign from $299 to $250. Change nothing else."}
reply = requests.post(f"https://{host}/mai/v1/images/edits", headers=headers, data=form, files=files, timeout=300)
```

**Rate limits.** Each deployment accepts a set number of requests a minute (`setup.sh` deploys 6). When you go over, the service answers **429** with a `Retry-After` header. Wait that long, then retry. Notebook 01 spaces its requests so it rarely sees one.

### 👀 Observe: what MAI-Image-2.6 did in our run

**Words on pictures** (1.5) and **other languages** (1.7). Text rendering is a strength. English was the strongest; have a native speaker check the rest.

<img src="./examples/01/05-launch-poster.webp" alt="AI-generated launch poster for the TrailMaster X4 tent with the headline HOME, ANYWHERE." width="220" />
<img src="./examples/01/07-ad-spanish.webp" alt="AI-generated Spanish-language ad for the tent." width="260" />
<img src="./examples/01/07-ad-japanese.webp" alt="AI-generated Japanese-language ad for the tent." width="260" />

**Precise edits** (1.12). Remove a stranger, fix a price, recolor the rainfly, one change at a time.

<img src="./examples/01/12-campsite-before.webp" alt="AI-generated campsite photo, before editing." width="245" />
<img src="./examples/01/12-edit-1-remove-stranger.webp" alt="The same campsite with the stranger removed." width="245" />
<img src="./examples/01/12-edit-2-new-price.webp" alt="The same campsite with the sign price changed to 250 dollars." width="245" />

**Web grounding** (1.14). Same prompt, grounding off and on: the grounded version reflects current conditions.

<img src="./examples/01/14-window-no-grounding.webp" alt="AI-generated window scene without web grounding." width="245" />
<img src="./examples/01/14-window-grounded.webp" alt="AI-generated window scene with web grounding." width="245" />

| What we measured | Our result | The lesson |
| --- | --- | --- |
| A 7-point brief with a clock, mugs and poles (1.10) | **6 out of 7** right; the watch time slipped | Score with a checklist, and run it more than once |
| `auto_aspect_ratio=True` on three different scenes (1.15) | It picked **1536 × 1536** every time | When the shape matters, ask for an exact size |
| Standard vs Flash, same prompts (1.16) | Flash took **~52%** of the time (14.6 s vs 28.1 s median) | Draft with Flash, finish with the standard model |

In [11]:
from statistics import median

by_model: dict[str, list[float]] = {}
for name, entry in image_log.items():
    by_model.setdefault(entry["model"], []).append(entry["seconds"])

print("Every picture in our run (each model drew different prompts and sizes):")
for model, times in sorted(by_model.items()):
    print(f"   {model:22} {len(times):>2} pictures · median {median(times):5.1f} s · fastest {min(times):5.1f} s · slowest {max(times):5.1f} s")

taste = {model: median(e["seconds"] for n, e in image_log.items() if n.startswith("16-taste") and e["model"] == model) for model in by_model}
print("\nThe fair test (1.16): same four prompts, same size, both models:")
for model, seconds in sorted(taste.items()):
    print(f"   {model:22} median {seconds:5.1f} s")


Every picture in our run (each model drew different prompts and sizes):
   MAI-Image-2.6          32 pictures · median  32.0 s · fastest  25.9 s · slowest  45.1 s
   MAI-Image-2.6-Flash    15 pictures · median  13.9 s · fastest  11.7 s · slowest  23.1 s

The fair test (1.16): same four prompts, same size, both models:
   MAI-Image-2.6          median  28.1 s
   MAI-Image-2.6-Flash    median  14.6 s


<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 4. Build with voice: MAI-Voice-2.1

<img src="./assets/asks/02-08.svg" alt="Your customer asks: Our setup hotline has to feel live. How fast does it start talking?" width="900" />

Voice uses the **Azure Speech SDK** on your resource's `cognitiveservices` address. There's nothing to deploy. You pick the model inside **SSML**, the markup that tells the speech service what to say and how.

### 🧱 Construct SSML, one layer at a time

| Layer | Markup | Why |
| --- | --- | --- |
| 1. Words | `TrailMaster X4. Home, anywhere.` | What to say, with special characters escaped |
| 2. Mood (optional) | `<mstts:express-as style="excited">…</mstts:express-as>` | How to say it |
| 3. Voice + model | `<voice name="en-US-Harper:MAI-Voice-2.1">…</voice>` | Who says it, and which MAI model |
| 4. Document | `<speak version="1.0" xmlns=… xml:lang="en-US">…</speak>` | Wraps one or more voices |

The `voice:model` pair in layer 3 is the switch between MAI-Voice-2.1 and MAI-Voice-2.1-Flash.

In [5]:
import xml.etree.ElementTree as ElementTree
from html import escape


def ssml(text: str, voice: str = "en-US-Harper", model: str = "MAI-Voice-2.1", style: str | None = None, lang: str = "en-US", safe: bool = True) -> str:
    body = escape(text) if safe else text
    if style:
        body = f'<mstts:express-as style="{style}">{body}</mstts:express-as>'
    return ('<speak version="1.0" xmlns="http://www.w3.org/2001/10/synthesis" '
            f'xmlns:mstts="http://www.w3.org/2001/mstts" xml:lang="{lang}">'
            f'<voice name="{voice}:{model}">{body}</voice></speak>')


hotline = ssml("Thanks for calling Contoso Outdoors. The TrailMaster X4 sleeps four.", model="MAI-Voice-2.1-Flash", style="customer_call_center")
ElementTree.fromstring(hotline)  # The speech service rejects SSML that isn't valid XML, so check it first.
print("✅ Valid SSML:\n" + hotline)

✅ Valid SSML:
<speak version="1.0" xmlns="http://www.w3.org/2001/10/synthesis" xmlns:mstts="http://www.w3.org/2001/mstts" xml:lang="en-US"><voice name="en-US-Harper:MAI-Voice-2.1-Flash"><mstts:express-as style="customer_call_center">Thanks for calling Contoso Outdoors. The TrailMaster X4 sleeps four.</mstts:express-as></voice></speak>


### 🐞 Debug: the ampersand that breaks a radio spot

🔮 **Predict:** marketing writes *"Rain & shine, it's home."* What happens if we drop it into SSML as is?

In [12]:
script = "TrailMaster X4. Rain & shine, it's home. Just $250."

for safe in (False, True):
    try:
        ElementTree.fromstring(ssml(script, safe=safe))
        print(f"✅ escaped={safe}: valid SSML, ready to speak")
    except ElementTree.ParseError as problem:
        print(f"🐞 escaped={safe}: {problem}. The SDK would cancel with an SSML error, or read silence.")

print("\n🔧 Fix: always escape() the words before they go into SSML. &, < and > are the usual suspects.")

🐞 escaped=False: not well-formed (invalid token): line 1, column 188. The SDK would cancel with an SSML error, or read silence.
✅ escaped=True: valid SSML, ready to speak

🔧 Fix: always escape() the words before they go into SSML. &, < and > are the usual suspects.


### 🧱 Construct: the real call

This is the heart of `speak()` in notebook 02:

```python
import azure.cognitiveservices.speech as speechsdk

resource = urlparse(FOUNDRY_PROJECT_ENDPOINT).hostname.split(".")[0]
config = speechsdk.SpeechConfig(token_credential=credential, endpoint=f"https://{resource}.cognitiveservices.azure.com/")
config.set_speech_synthesis_output_format(speechsdk.SpeechSynthesisOutputFormat.Audio24Khz48KBitRateMonoMp3)

synthesizer = speechsdk.SpeechSynthesizer(speech_config=config, audio_config=speechsdk.audio.AudioOutputConfig(filename="hotline.mp3"))
result = synthesizer.speak_ssml_async(hotline).get()
if result.reason != speechsdk.ResultReason.SynthesizingAudioCompleted:
    print(result.cancellation_details.error_details)   # The real reason: auth, region, voice or SSML
```

To measure **time to first sound**, connect to `synthesizer.synthesizing` and note when the first chunk arrives. That's what a caller feels.

### 👀 Observe: what MAI-Voice-2.1 did in our run

| What we measured (notebook 02) | MAI-Voice-2.1 | MAI-Voice-2.1-Flash |
| --- | --- | --- |
| First sound, hotline answer (median of 3, after a warm-up) | **1.49 s** | **1.12 s** |
| Whole answer ready | 1.60 s | 1.27 s |
| Radio spot, 14.3 s of audio | first sound after 3.59 s (first call, cold) | — |
| Setup guide read aloud, 84 s of audio | first sound after 1.76 s | — |

**Back to your prediction.** The model page quotes ~550 ms vs ~45 ms of *model inference*: about 12 times faster. Our callers heard the first word about **25% sooner**, not 12 times. The rest of the wait is the network, sign-in and the SDK, and it's the same for both models.

> 💡 **The lesson:** vendor latency is model inference. Your users feel end-to-end latency. Measure it in your region, from your code, and throw away the first (cold) call, as notebook 02 does in section 2.8.

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 5. Build with transcription: MAI-Transcribe-2

<img src="./assets/asks/03-05.svg" alt="Your customer asks: Can we trust a transcript enough to act on it?" width="900" />

Transcription is one REST call to your resource's `cognitiveservices` address. You send the audio and a small JSON **definition**; MAI-Transcribe-2 is chosen inside it.

### 🧱 Construct the definition, one feature at a time

| Feature | Add to the definition | When |
| --- | --- | --- |
| Use MAI-Transcribe-2 | `"enhancedMode": {"enabled": true, "model": "MAI-Transcribe-2"}` | Always |
| Keyword hints | `"phraseList": {"phrases": ["TrailMaster X4", "rainfly"]}` | Product names, jargon |
| Speaker labels | `"diarization": {"enabled": true}` | Calls and meetings |

In [7]:
def transcribe_definition(model: str = "MAI-Transcribe-2", hints: list[str] | None = None, speakers: bool = False) -> dict:
    definition: dict = {"enhancedMode": {"enabled": True, "model": model}}
    if hints:
        definition["phraseList"] = {"phrases": hints}
    if speakers:
        definition["diarization"] = {"enabled": True}  # MAI-Transcribe-2 rejects maxSpeakers for now, so don't set it
    return definition


print(json.dumps(transcribe_definition(hints=["TrailMaster X4", "OutdoorLiving", "rainfly"], speakers=True), indent=2))

{
  "enhancedMode": {
    "enabled": true,
    "model": "MAI-Transcribe-2"
  },
  "phraseList": {
    "phrases": [
      "TrailMaster X4",
      "OutdoorLiving",
      "rainfly"
    ]
  },
  "diarization": {
    "enabled": true
  }
}


### 🧱 Construct: the real call

```python
API_VERSION = "2025-10-15"
reply = requests.post(
    f"https://{resource}.cognitiveservices.azure.com/speechtotext/transcriptions:transcribe?api-version={API_VERSION}",
    headers=headers, timeout=300,
    files={"audio": ("radio.mp3", open("radio.mp3", "rb"), "audio/mpeg"),
           "definition": (None, json.dumps(definition), "application/json")})
text = " ".join(phrase["text"] for phrase in reply.json()["combinedPhrases"])
```

### 🐞 Debug: is the transcript wrong, or is your scoring wrong?

**Word error rate (WER)** counts how many words you'd have to change to fix the transcript, divided by the number of words that were really said. 0 is perfect.

🔮 **Predict:** the script says *"Just $250."* and the transcript says *"Just two hundred fifty dollars."* Is that a mistake?

In [8]:
import difflib
import re
import unicodedata


def words(text: str) -> list[str]:
    """The scoring rules, decided before we look at any score: lowercase, no accents, no punctuation."""
    text = unicodedata.normalize("NFKD", text.lower())
    return re.findall(r"[a-z0-9$]+", "".join(c for c in text if not unicodedata.combining(c)))


def wer(expected: str, heard: str) -> float:
    want, got = words(expected), words(heard)
    changes = sum(max(a2 - a1, b2 - b1) for op, a1, a2, b1, b2 in difflib.SequenceMatcher(a=want, b=got, autojunk=False).get_opcodes() if op != "equal")
    return changes / max(len(want), 1)


script = "The TrailMaster X4 sleeps four. Just $250."
for heard in ("The TrailMaster X4 sleeps four. Just $250.",
              "The TrailMaster X4 sleeps four. Just two hundred fifty dollars.",
              "The trail master X four sleeps four. Just $250."):
    print(f"WER {wer(script, heard):.2f} · {heard}")

print("\n🔧 Both 'mistakes' are the same words, written differently. Decide your rules for numbers and names first, then score.")

WER 0.00 · The TrailMaster X4 sleeps four. Just $250.
WER 0.57 · The TrailMaster X4 sleeps four. Just two hundred fifty dollars.
WER 0.57 · The trail master X four sleeps four. Just $250.

🔧 Both 'mistakes' are the same words, written differently. Decide your rules for numbers and names first, then score.


### 🧱 Construct: read the reply

The [model card](../../.github/.mc/MAI-Transcribe-2-Model-Card.pdf) shows the shape. `combinedPhrases` is the whole transcript; `phrases` gives each piece with its speaker and timing, which is what you search and subtitle with:

```json
{
  "durationMilliseconds": 4000,
  "combinedPhrases": [{ "text": "Gotcha.\nThat makes sense." }],
  "phrases": [
    { "speaker": 0, "offsetMilliseconds": 80,  "durationMilliseconds": 339, "text": "Gotcha.",           "locale": "en" },
    { "speaker": 1, "offsetMilliseconds": 800, "durationMilliseconds": 460, "text": "That makes sense.", "locale": "en" }
  ]
}
```

```python
for phrase in reply.json()["phrases"]:
    print(f"{phrase['offsetMilliseconds'] / 1000:6.1f}s  speaker {phrase.get('speaker', '?')}: {phrase['text']}")
```

### 👀 Observe: what MAI-Transcribe-2 did in our run

| What we measured (notebook 03) | Our result | The lesson |
| --- | --- | --- |
| English radio spot (14 s) | Transcribed in **0.73 s**, WER **0.00** | Short clips come back almost instantly |
| Spanish and French spots, language not given | Detected **es** and **fr** correctly · WER **0.08** and **0.11** | Detection works; non-English scores are lower, so have a native speaker review |
| Setup guide, with and without keyword hints | Same WER (0.00) both ways | Hints matter on real, noisy audio with unusual names. Clean synthetic audio is too easy to show it |
| Two-voice support call, speaker labels on | Transcribed in **1.5 s**, speakers separated | Label speakers for calls, then search by who said what |

> ⚠️ **The honest limit:** our audio came from MAI-Voice, so it's studio-clean. Real calls have accents, noise and people talking over each other. Build your answer key from **your** recordings before you quote an accuracy number.

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 6. Beyond your resource: Code, Thinking and Cyber

Three MAI models don't go through the three doors. You meet them elsewhere.

### 💻 MAI-Code-1.1-Flash, in GitHub Copilot

A lightweight agentic coding model, custom-trained for VS Code and the Copilot CLI. It plans and drives work from idea to implementation, reads **screenshots, diagrams and mock-ups**, and on supported devices can run **locally**, with no per-call charges.

**How you use it:** in Copilot Chat, open the **model picker**, choose MAI-Code-1.1-Flash, and switch to **Agent** mode. There's no API key and nothing to deploy. Notebook 04 hands it a napkin sketch:

<img src="./assets/napkin/inked.webp" alt="The hand-drawn napkin sketch of the new storefront." width="300" />
<img src="./examples/04/2-wireframe-approved.svg" alt="The wireframe the model drew from the napkin, after a person approved it." width="420" />

**🧱 Construct a coding prompt that works** (notebook 04, section 4.12):

1. **One step per prompt.** Read the sketch, then approve a wireframe, then build.
2. **Name the source of truth.** *"Use only `data/contoso-retail/products.json`."*
3. **Say what not to do.** *"Don't invent products, prices or features."*
4. **Ask it to check its work.** *"Run `npm run build` and fix any errors."*

🐞 **Debug:** the catalog shows 21 products, but the data has 20. Don't fix the page. Ask: *"List anything on the site that isn't backed by `data/contoso-retail/`."* Then remove what it finds.

### 🧠 MAI-Thinking-1, in Microsoft Foundry

Microsoft AI's first reasoning model: a mixture-of-experts model (about 35B active and ~1T total parameters), built for math, coding and enterprise work, in **public preview** in Foundry. Use it where reasoning earns its cost, such as planning, analysis and hard code reviews, not for every reply. Start with the [MAI-Thinking how-to guide](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-mai-thinking).

### 🛡️ MAI-Cyber-1.1-Flash, through MDASH

A cybersecurity model inside **MDASH**, Microsoft's multi-agent system that finds, validates and fixes vulnerabilities. It handles most MDASH tasks and saves the largest models for the hardest ones. It's **available only to verified defenders through MDASH**, so there's no direct API to call. See the [model page](https://microsoft.ai/models/mai-cyber-1-1-flash/) to register interest.

> 🎛️ **Need a model tuned to your own data?** Microsoft AI also offers [Microsoft Frontier Tuning](https://microsoft.ai/models/microsoft-frontier-tuning/), a custom fine-tuning service.

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 7. The debugging playbook

When a call fails, read **two things**: the status code, and the service's own message. Notebooks 01 to 03 print both, with a hint.

### By status code

| You see | It usually means | Try |
| --- | --- | --- |
| **400** (images) | A size or option the service won't take | Each side ≥ 768, total ≤ 2,359,296 pixels; references must be PNG or JPEG |
| **401** | No valid sign-in | `az login --use-device-code`, then rerun the cell |
| **403** | Signed in, but no permission **yet** | Check the **Foundry User** role. Right after `setup.sh`, wait a few minutes: new role assignments take time to apply |
| **404** (images) | No deployment with that name | Compare `MODEL_IMAGE_HERO` / `MODEL_IMAGE_BULK` in `.env` with **Models + endpoints** in the Foundry portal |
| **404** (speech) | Wrong address, or the region lacks that model | Use a region with MAI voices and MAI-Transcribe, such as `eastus` |
| **429** | Over the rate limit | Wait for `Retry-After` seconds, then retry. Space your requests out |
| **5xx** | The service is busy | Retry with a pause. Notebook 01 retries up to four times |

### By symptom

| You see | It usually means | Try |
| --- | --- | --- |
| Speech *"didn't finish"* (canceled) | Auth, region, voice name or bad SSML | Print `result.cancellation_details.error_details`. Validate your SSML as in section 4 |
| Silence, or a very short MP3 | Empty or broken SSML | Print the SSML and read it |
| The wrong language, in the right voice | The voice name's locale doesn't match the text | Change the prefix: `es-MX-Harper:MAI-Voice-2.1` for Spanish |
| A cloned voice is refused | Voice cloning is gated | Apply for limited access, and upload the speaker's recorded consent (see the [Voice card](../../.github/.mc/MAI-Voice-2.1-Model-Card-Memo.pdf)) |
| An empty transcript | No audio in the file, or a format it can't read | Play the file first. Send WAV, MP3 or FLAC |
| A long recording is rejected | Over the batch limit | Split files over **300 MB or 2 hours** |
| Live captions don't work in `eastus` | MAI-Transcribe-2-Streaming isn't served there | Use a Streaming region: Central US, Sweden Central or Southeast Asia |
| Replay says *"no saved result"* | That cell has no saved example | Set `DEMO_MODE="live"` to make your own |
| A website with a product you don't sell | The coding model guessed | Name the data source in the prompt, and check the result against it |
| MAI-Code isn't in the model picker | Rollout or plan | Check your Copilot plan; the model may also be chosen for you through **Auto** |

**Back to the prediction in section 2:** a 403 straight after `setup.sh` usually isn't a broken deployment. Your role assignment is still spreading. Wait a few minutes, then retry.

### 🐞 Run the checklist

This cell checks your setup without calling any model, and never prints a secret.

In [9]:
import os
import shutil
import subprocess

from dotenv import load_dotenv

load_dotenv(HERE / ".env")


def check(label: str, ok: bool, fix: str) -> None:
    print(f"{'✅' if ok else '⚠️ '} {label}" + ("" if ok else f"\n   🔧 {fix}"))


az = shutil.which("az")
signed_in = bool(az) and subprocess.run([az, "account", "show"], capture_output=True).returncode == 0
check("Azure CLI installed", bool(az), "Use the dev container, or install the Azure CLI.")
check("Signed in to Azure", signed_in, "Run: az login --use-device-code")
check(".env exists in this folder", (HERE / ".env").exists(), "Run: bash infra/setup.sh  (or cp .env.example .env)")
for name in ("FOUNDRY_PROJECT_ENDPOINT", "MODEL_IMAGE_HERO", "MODEL_IMAGE_BULK"):
    value = os.environ.get(name, "").strip()
    check(f"{name} is set", bool(value) and "<" not in value, "Run: bash infra/setup.sh, which writes it.")
endpoint = os.environ.get("FOUNDRY_PROJECT_ENDPOINT", "")
check("Endpoint has the expected shape", not endpoint or ".services.ai.azure.com/api/projects/" in endpoint,
      "It should look like https://<name>.services.ai.azure.com/api/projects/<project>")
print("\nAll ✅? Switch a notebook to live. Any ⚠️? Follow the 🔧, then run this cell again.")

✅ Azure CLI installed
✅ Signed in to Azure
⚠️  .env exists in this folder
   🔧 Run: bash infra/setup.sh  (or cp .env.example .env)
⚠️  FOUNDRY_PROJECT_ENDPOINT is set
   🔧 Run: bash infra/setup.sh, which writes it.
⚠️  MODEL_IMAGE_HERO is set
   🔧 Run: bash infra/setup.sh, which writes it.
⚠️  MODEL_IMAGE_BULK is set
   🔧 Run: bash infra/setup.sh, which writes it.
✅ Endpoint has the expected shape

All ✅? Switch a notebook to live. Any ⚠️? Follow the 🔧, then run this cell again.


<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 8. Pick the right model

<img src="./assets/asks/01-16.svg" alt="Your customer asks: Do we pay for the best model every time, or is there a cheaper option?" width="900" />

The question every notebook asks: **which of our jobs actually need the big one?**

| The job | Pick | Because |
| --- | --- | --- |
| A hero poster people will study | **MAI-Image-2.6** | Quality is the point |
| Twenty drafts to choose from | **MAI-Image-2.6-Flash**, then finish the winner with 2.6 | About half the time per picture in our run, at less than half the price |
| A radio ad or an audiobook | **MAI-Voice-2.1** | Fidelity matters more than speed, and it keeps long-form consistent |
| A phone line or voice agent | **MAI-Voice-2.1-Flash** | First sound came ~25% sooner in our run, it's cheaper per character, and callers can interrupt it |
| Nightly call transcripts | **MAI-Transcribe-2** | Batch is fast and the cheapest per hour, with speaker labels and timestamps |
| Live captions | **MAI-Transcribe-2-Streaming** | First words in ~120 ms. Deploy in a Streaming region, not `eastus` |
| A website from a sketch | **MAI-Code-1.1-Flash** in Copilot | Trained in the Copilot harness; short, scoped prompts suit a Flash model |
| A hard planning or analysis step | **MAI-Thinking-1** | Reasoning earns its cost here, not on every reply |

### 🧮 Work out the bill

Prices from the model cards and pages when this guide was written. The MAI-Transcribe-2 price is a limited-time offer, and the Streaming price runs through 31 December 2026. Check the [MAI pricing page](https://aka.ms/MAIModelPricing) before you quote them.

In [10]:
VOICE_USD_PER_MILLION_CHARS = {"MAI-Voice-2.1": 22.0, "MAI-Voice-2.1-Flash": 15.0}
TRANSCRIBE_USD_PER_HOUR = {"MAI-Transcribe-2": 0.10, "MAI-Transcribe-2-Streaming": 0.54}

hotline_answer = "Thanks for calling Contoso Outdoors. The TrailMaster X4 sleeps four, and it's water-resistant with a rainfly."
calls_per_month = 50_000
characters = len(hotline_answer) * calls_per_month

print(f"🎙️ {calls_per_month:,} hotline answers of {len(hotline_answer)} characters = {characters:,} characters a month")
for model, price in VOICE_USD_PER_MILLION_CHARS.items():
    print(f"   {model:22} ${characters / 1_000_000 * price:8.2f} a month")

hours = 2_000
print(f"\n👂 {hours:,} hours of support calls a month")
for model, price in TRANSCRIBE_USD_PER_HOUR.items():
    print(f"   {model:28} ${hours * price:8.2f} a month")

🎙️ 50,000 hotline answers of 109 characters = 5,450,000 characters a month
   MAI-Voice-2.1          $  119.90 a month
   MAI-Voice-2.1-Flash    $   81.75 a month

👂 2,000 hours of support calls a month
   MAI-Transcribe-2             $  200.00 a month
   MAI-Transcribe-2-Streaming   $ 1080.00 a month


<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 9. Check yourself

1. You call the image API with `"model": "MAI-Image-2.6"` and get **404**. The model is in your region. What's the most likely cause?
2. Why does a 1600 × 1600 picture fail, when 2304 × 1024 works?
3. How do you switch a voice from MAI-Voice-2.1 to Flash? And how do you make the same voice speak Spanish?
4. The model page says Flash is ~12 times faster. Why did our callers hear the first word only ~25% sooner?
5. A transcript says *"two hundred fifty dollars"* where the script said *"$250"*. Is the model wrong?
6. Where do you use MAI-Code-1.1-Flash, and why is there no key?
7. Your app adds live captions, using the same `eastus` resource as the workshop. Why might it fail?
8. A customer wants MAI-Cyber-1.1-Flash for a penetration-testing tool. What do you tell them?

<details>
<summary>Show answers</summary>

1. `model` must be your **deployment name**. If you named the deployment something else, use that name.
2. The limit is total pixels, not the longest side: 1600 × 1600 = 2,560,000, over 2,359,296. 2304 × 1024 is exactly at the limit.
3. Change the model after the colon: `en-US-Harper:MAI-Voice-2.1-Flash`. For Spanish, change the locale before it: `es-MX-Harper:MAI-Voice-2.1`. It's the same voice in every language.
4. The vendor number is model inference. The network, sign-in and SDK add the same time to both, so end-to-end the gap is smaller.
5. No. It's the same words written differently. Decide your scoring rules for numbers and names before you score.
6. In GitHub Copilot's model picker, in VS Code. It's covered by your Copilot plan, so you never handle a key.
7. Live captions need MAI-Transcribe-2-Streaming, which its model card lists in Central US, Sweden Central and Southeast Asia, not East US.
8. It's only available inside MDASH, for verified defenders, and only for defensive security work. There's no direct API.

</details>

### Where to go next

| You want to… | Go to |
| --- | --- |
| See the whole launch story | [00 · Set the stage](./00-set-the-stage.ipynb) |
| Run images, voice and transcription live | [01](./01-build-launch-kit.ipynb), [02](./02-give-it-a-voice.ipynb), [03](./03-hear-your-customers.ipynb), after `bash infra/setup.sh` |
| Build the storefront with Copilot | [04 · Refresh the storefront](./04-refresh-the-storefront.ipynb) |
| Reuse the product data | [Contoso Retail dataset](../../data/contoso-retail/README.md) |
| Read the model cards | [Model cards (PDF)](../../.github/.mc/) |
| Read the model pages | [microsoft.ai/models](https://microsoft.ai/models/) |

➡️ Back to the [workshop README](./README.md).